## Limpieza de 'DF_CHAMPIONSXIP_SUCIO.csv'

Cargamos la tabla de carreras/modalidades (una fila por cada modalidad de cada evento) y le hacemos una primera inspección antes de limpiarla.

In [1]:
from pathlib import Path
import pandas as pd

CSV_PATH = Path("../../data/raw/championsxip/DF_CHAMPIONSXIP_SUCIO.csv")
curses = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

print("Filas x columnas:", curses.shape)
print()
print(curses.dtypes)
curses.head()

Filas x columnas: (2727, 8)

Nom_Cursa             object
Data                  object
Modalitat             object
Distancia_Subcursa    object
Finishers_Homes        int64
Finishers_Dones        int64
URL_Resultats         object
Total_Finishers        int64
dtype: object


,Nom_Cursa,Data,Modalitat,Distancia_Subcursa,Finishers_Homes,Finishers_Dones,URL_Resultats,Total_Finishers
0,EWR WINTER EDITION TERRASSA,31/01/2021,trail,EWR Winter Edition,0,0,https://xipgroc.cat/ca/curses/EWRWinter2020/ew...,0
1,10A CURSA DE CAP D’ANY DE SABADELL - VIRTUAL,31/01/2021,running,5-10K,0,0,https://xipgroc.cat/ca/curses/capdany2020/5-10...,0
2,TEST CRONOMETRAT 5K,21/02/2021,running-pista.,Sèries,103,25,https://xipgroc.cat/ca/curses/Marbella_5K_01/s...,128
3,ULTRARUNNING BARCELONA,13/03/2021,running-pista.,10K,18,3,https://xipgroc.cat/ca/curses/Ultrarunning2021...,21
4,ULTRARUNNING BARCELONA,13/03/2021,running-pista.,20K,18,3,https://xipgroc.cat/ca/curses/Ultrarunning2021...,21


In [2]:
# Diagnóstico antes de limpiar: nulos, duplicados y consistencia de conteos
print("Valores nulos por columna:")
print(curses.isna().sum())
print()

print("Filas completamente duplicadas:", curses.duplicated().sum())
print("Filas duplicadas por (Nom_Cursa, Data, Distancia_Subcursa):",
      curses.duplicated(subset=["Nom_Cursa", "Data", "Distancia_Subcursa"]).sum())
print()

_fecha = pd.to_datetime(curses["Data"], format="%d/%m/%Y", errors="coerce")
print("Fechas que no se han podido parsear:", _fecha.isna().sum())
print("Rango de fechas:", _fecha.min(), "->", _fecha.max())
print()

print("Finishers_Homes + Finishers_Dones vs Total_Finishers, deberían coincidir:")
suma = curses["Finishers_Homes"] + curses["Finishers_Dones"]
print((suma != curses["Total_Finishers"]).sum(), "filas no coinciden")
print()

print("Valores de 'Modalitat':")
print(curses["Modalitat"].value_counts())

Valores nulos por columna:
Nom_Cursa             0
Data                  0
Modalitat             0
Distancia_Subcursa    0
Finishers_Homes       0
Finishers_Dones       0
URL_Resultats         0
Total_Finishers       0
dtype: int64

Filas completamente duplicadas: 95
Filas duplicadas por (Nom_Cursa, Data, Distancia_Subcursa): 96

Fechas que no se han podido parsear: 0
Rango de fechas: 2021-01-31 00:00:00 -> 2026-08-01 00:00:00

Finishers_Homes + Finishers_Dones vs Total_Finishers, deberían coincidir:
0 filas no coinciden

Valores de 'Modalitat':
Modalitat
trail             1337
running           1308
running-pista.      82
Name: count, dtype: int64


In [3]:
# A diferencia de buscametas (sin duplicados exactos), aquí sí hay filas
# completamente duplicadas (mismo evento, misma subcursa, mismos
# finishers) — las quitamos antes de seguir, igual que en carreirasgalegas.
antes = len(curses)
curses = curses.drop_duplicates().reset_index(drop=True)
print(f"{antes - len(curses)} filas duplicadas eliminadas ({antes} -> {len(curses)})")

95 filas duplicadas eliminadas (2727 -> 2632)


In [4]:
# Limpieza: nos quedamos con las columnas que interesan, renombradas.
# No hay municipio/provincia/id como en buscametas (esta fuente no los
# ofrece); URL_Resultats es un artefacto técnico del scraping y
# Total_Finishers es redundante (= Finishers_Homes + Finishers_Dones, ya
# comprobado arriba), así que tampoco se incluyen. finisher_d/finisher_h
# (mismo nombre que en el resto de fuentes, antes finished_d/finished_h).
curses_limpio = curses[
    ["Nom_Cursa", "Data", "Modalitat", "Distancia_Subcursa", "Finishers_Dones", "Finishers_Homes"]
].rename(columns={
    "Nom_Cursa": "nombre_carrera",
    "Data": "fecha",
    "Finishers_Dones": "finisher_d",
    "Finishers_Homes": "finisher_h",
})

curses_limpio["fecha"] = pd.to_datetime(curses_limpio["fecha"], format="%d/%m/%Y")

print(curses_limpio.shape)
curses_limpio.head()

(2632, 6)


,nombre_carrera,fecha,Modalitat,Distancia_Subcursa,finisher_d,finisher_h
0,EWR WINTER EDITION TERRASSA,2021-01-31,trail,EWR Winter Edition,0,0
1,10A CURSA DE CAP D’ANY DE SABADELL - VIRTUAL,2021-01-31,running,5-10K,0,0
2,TEST CRONOMETRAT 5K,2021-02-21,running-pista.,Sèries,25,103
3,ULTRARUNNING BARCELONA,2021-03-13,running-pista.,10K,3,18
4,ULTRARUNNING BARCELONA,2021-03-13,running-pista.,20K,3,18


In [5]:
# A diferencia de buscametas, aquí la disciplina ya viene dada
# directamente por 'Modalitat' (solo 3 valores posibles), así que no hace
# falta clasificarla por palabras clave — solo mapearla al mismo esquema
# de categorías (trail running / road running) que usamos en el resto de
# fuentes. "running-pista." (atletismo en pista) se agrupa con road
# running.
_MODALITAT_A_TIPO = {
    "trail": "trail running",
    "running": "road running",
    "running-pista.": "road running",
}
curses_limpio["tipo_modalidad"] = curses_limpio["Modalitat"].map(_MODALITAT_A_TIPO)
curses_limpio = curses_limpio.drop(columns=["Modalitat"])

print(curses_limpio["tipo_modalidad"].value_counts())

tipo_modalidad
road running     1332
trail running    1300
Name: count, dtype: int64


In [6]:
# "Distancia_Subcursa" mezcla dos cosas, igual que "modalidad" en
# buscametas: a veces es la distancia (5K, 10K, 3000m...) y a veces es la
# categoría/edad (SB8MF, M2015, MASTER_MF...). Extraemos primero la
# distancia (en km) de los formatos numéricos (K/Km, m/metros, con coma o
# guión bajo como separador decimal) y de los nombres conocidos sin
# número (Mitja=media maratón, Marató=maratón, Milla).
import re

_km = curses_limpio["Distancia_Subcursa"].str.extract(r"^(\d+(?:[.,_]\d+)?)\s*[kK][mM]?\b")[0]
distancia_km = _km.str.replace(",", ".", regex=False).str.replace("_", ".", regex=False).astype(float)

_m = curses_limpio["Distancia_Subcursa"].str.extract(
    r"^(\d+(?:[.,]\d+)?)\s*m(?:ts|etros)?\.?\b", flags=re.IGNORECASE
)[0]
distancia_m = _m.str.replace(",", ".", regex=False).astype(float) / 1000

curses_limpio["distancia"] = distancia_km
_falta = curses_limpio["distancia"].isna()
curses_limpio.loc[_falta, "distancia"] = distancia_m[_falta]

_falta = curses_limpio["distancia"].isna()
_es_mitja = curses_limpio["Distancia_Subcursa"].str.contains(r"mitja", case=False, na=False)
_es_marato = curses_limpio["Distancia_Subcursa"].str.contains(r"marat", case=False, na=False)
_es_milla = curses_limpio["Distancia_Subcursa"].str.contains(r"milla", case=False, na=False)
curses_limpio.loc[_falta & _es_mitja, "distancia"] = 21.097
curses_limpio.loc[_falta & _es_marato, "distancia"] = 42.195
curses_limpio.loc[_falta & _es_milla, "distancia"] = 1.609

curses_limpio["distancia"] = curses_limpio["distancia"].fillna(0)

print("Filas con distancia detectada:", (curses_limpio["distancia"] != 0).sum(),
      "de", len(curses_limpio))
print()
print("Ejemplos de Distancia_Subcursa SIN distancia detectada (revisa si falta algún patrón):")
print(curses_limpio.loc[curses_limpio["distancia"] == 0, "Distancia_Subcursa"].value_counts().head(30))

Filas con distancia detectada: 956 de 2632

Ejemplos de Distancia_Subcursa SIN distancia detectada (revisa si falta algún patrón):
Distancia_Subcursa
SB8MF        34
M2015        30
M2016        30
SB12F        28
SB10MF       27
F2015        27
SB12MF       27
SB14MF       27
SB16MF       27
F2016        27
SB12M        26
MASTER_MF    26
SB10F        25
ABS_MF       25
SB10M        24
M2014        24
MF2012       23
M2017        23
SB14F        22
SB14M        21
F2014        21
F2017        20
MF2009       17
MF2011       17
MF2010       17
M2018        16
MF2013       16
SB10         15
SB12         15
MF2017       15
Name: count, dtype: int64


In [7]:
# Igual que en buscametas quitábamos el "nK" de modalidad antes de
# clasificar, aquí quitamos el trozo de distancia ya extraído de
# "Distancia_Subcursa" para quedarnos solo con el texto de categoría
# (SB8MF, M2015, CadMF...) de cara a clasificar el público.
_categoria = (
    curses_limpio["Distancia_Subcursa"]
    .str.replace(r"^\d+(?:[.,_]\d+)?\s*[kKmM][mM]?\.?\b", "", regex=True)
    .str.replace(r"[_/]", " ", regex=True)
    .str.strip()
)

# Clasificamos el público (edad) por palabras clave, con las
# abreviaturas habituales del atletismo catalán: preB/Ben/Ale/Inf/Cad/Juv
# (todas se tratan como "Infantil" — Cadete/Juvenil es la misma categoría
# que Infantil, ver Limpieza_union.ipynb), Vet/Master (veteranos). Los
# códigos "SBxx"/"SUBxx" llevan directamente la edad en el número. Antes
# de nada comprobamos si es una carrera especial (con discapacidad,
# handbike, divisió funcional...) o de élite/profesional — eso va a
# "Otros"/"Elite" porque no es una cuestión de edad. Si no hay ninguna
# marca de edad ni es una carrera especial, asumimos Absoluta/General por
# defecto: la inmensa mayoría de lo que no lleva categoría de edad es
# justamente eso (una carrera/10K sin categoría, "Pop"/popular, "Open",
# "CR"...).
_EQUIPOS_PATRON = r"equipos?\b|equips?\b"

def _clasificar_publico(texto, nombre_carrera=None):
    t = "" if pd.isna(texto) else texto.lower().strip()
    texto_nombre = "" if pd.isna(nombre_carrera) else nombre_carrera.lower()

    # "Equipos" manda por encima de cualquier otra cosa, igual que en ccnorte.
    if re.search(_EQUIPOS_PATRON, texto_nombre) or re.search(_EQUIPOS_PATRON, t):
        return "Equipos"

    if re.search(r"discap|invident|handbike|cadira de rodes|div.?funcional", t):
        return "Otros"

    if re.search(r"\belit|profession|profesional", t):
        return "Elite"

    if re.search(r"vet|\bvt\b|master|mast|^[mf]\s?\d{2}(?:[\s-]\d{2})?$", t):
        return "Mayores/Veteranos"

    _sb = re.search(r"\bsu?b\s?-?(\d{1,2})(?!\d)", t)
    if _sb:
        edad = int(_sb.group(1))
        if edad <= 23:
            return "Infantil"

    if re.search(
        r"preb|\bpb\b|\bben\b|benjam|\bale\b|alevi|\binf|infantil|escolar|"
        r"cadet|\bcad(?!ir)|juvenil|\bjuv|promesa",
        t,
    ):
        return "Infantil"

    # Categorías por año de nacimiento (M2015, F2016, MF2012...) — en
    # este rango de años son siempre categorías infantiles/de base.
    if re.search(r"^[mf]f?\s?(19|20)\d{2}\b", t):
        return "Infantil"

    return "Absoluta/General"

curses_limpio["publico"] = [
    _clasificar_publico(c, n) for c, n in zip(_categoria, curses_limpio["nombre_carrera"])
]

print(curses_limpio["publico"].value_counts())
print()
print("Texto de categoría clasificado como Otros (carreras especiales):")
print(_categoria[curses_limpio["publico"] == "Otros"].value_counts())

publico
Infantil             1308
Absoluta/General     1223
Mayores/Veteranos      71
Equipos                23
Elite                   4
Otros                   3
Name: count, dtype: int64

Texto de categoría clasificado como Otros (carreras especiales):
Distancia_Subcursa
SUB10M Discap.M    1
SUB10F Discap.F    1
Div Funcional      1
Name: count, dtype: int64


In [8]:
# "Distancia_Subcursa" ya ha cumplido su función: la hemos reclasificado
# en distancia y publico, así que la quitamos (igual que buscametas
# quita "modalidad" al final).
curses_limpio = curses_limpio.drop(columns=["Distancia_Subcursa"])
curses_limpio.columns.tolist()

['nombre_carrera',
 'fecha',
 'finisher_d',
 'finisher_h',
 'tipo_modalidad',
 'distancia',
 'publico']

### Ubicación (opcional) — igual que hicimos en ccnorte

`DF_CHAMPIONSXIP_SUCIO.csv` no da ninguna ubicación geográfica (a diferencia de buscametas, que sí trae `municipi`/`comarca_provincia`). Como alternativa, igual que en `Scraper_ccnorte.ipynb`, intentamos **extraer el lugar del propio `nombre_carrera`** con una heurística (quita numerales/ordinales, año, texto entre paréntesis y palabras genéricas del tipo de carrera — "CURSA POPULAR DE LA GARRIGA" -> "LA GARRIGA") y lo geocodificamos con **Nominatim/OpenStreetMap** (gratis, sin API key).

**Aviso — esto es un best-effort, no siempre acierta:** nombres de patrocinadores ("EWR WINTER EDITION TERRASSA"), lugares/instalaciones sin geocodificar en OSM ("ILLA CARLEMANY"), o títulos sin ningún lugar reconocible, pueden dar una ubicación incorrecta o ninguna. Revisa `championsxip_ubicaciones.csv` (columna `candidato_lugar`) antes de confiar en `municipio`/`provincia`.

Requiere `pip install geopy`. Nominatim limita a 1 petición/segundo — con ~500 nombres de carrera únicos tarda unos 10 minutos; tiene checkpoint propio (`championsxip_ubicaciones.csv`), se puede interrumpir y continuar.

In [9]:
# Heurística best-effort para aislar el lugar dentro del nombre de la
# carrera (adaptada del catalán: numerales "14A"/"57È"/"2N"/"3R", "de/d'/
# del/dels/a/en/al" como preposición de lugar). A diferencia de la versión
# de ccnorte (que corta SIEMPRE por la ÚLTIMA preposición), aquí cortamos
# por la PRIMERA preposición que aparece justo después de las palabras
# genéricas iniciales — así "CROS ESCOLAR DE CORNELLÀ DE LLOBREGAT" no
# pierde "Cornellà" y queda "CORNELLÀ DE LLOBREGAT" completo, no solo
# "LLOBREGAT" (muy habitual en Catalunya: Cornellà/Sant Boi/Sant Feliu...
# "de Llobregat", Santa Coloma "de Gramenet", etc.).
_RE_NUMERAL_INICIAL = re.compile(
    r"^\s*(?:[ivxlcdm]+|\d+)\s*[aeéè]?\.?(?:r|n|t|na|rt)?\s*[ºªo\.]*\s*[-–]?\s+",
    re.IGNORECASE,
)
_RE_ANIO = re.compile(r"\b(19|20)\d{2}\b")
_RE_DISTANCIA_FINAL = re.compile(r"\s+\d+(?:[.,]\d+)?\s?k(?:m)?\.?$", re.IGNORECASE)
_RE_DE_CONTRACTA = re.compile(r"^d['’]\s*", re.IGNORECASE)

_PALABRAS_GENERICAS = {
    "cursa", "carrera", "correr", "cros", "trail", "marato", "marató",
    "mitja", "milla", "milles", "popular", "urbana", "urbà", "urban",
    "semi", "escolar", "memorial", "trofeu", "trofeo", "circuit",
    "circuito", "campionat", "campeonato", "duatlo", "duatló", "triatlo",
    "triatló", "nocturna", "nocturn", "solidaria", "solidària", "solidari",
    "btt", "ruta", "gran", "premi", "premio", "edicio", "edició", "pujada",
    "volta", "copa", "lliga", "liga", "internacional", "provincial",
    "final", "comarcal", "fase", "previa", "prèvia", "cta", "cto", "xips",
    "xip", "groc", "run", "running", "race", "night", "ciutat", "vila",
    "poble", "festa", "festes", "fires", "aniversari", "diada",
}
_PREPOSICIONES = {"de", "d'", "del", "dels", "a", "en", "al"}


def _candidato_lugar(nombre_carrera: str) -> str:
    if not isinstance(nombre_carrera, str) or not nombre_carrera.strip():
        return ""

    texto = nombre_carrera.strip()
    texto = _RE_NUMERAL_INICIAL.sub("", texto)
    texto = _RE_ANIO.split(texto)[0]
    texto = texto.split("(")[0]
    texto = _RE_DISTANCIA_FINAL.sub("", texto)
    texto = texto.strip(" -–,.")

    tokens = texto.split()
    i = 0
    while i < len(tokens) and tokens[i].lower().strip(",.-") in _PALABRAS_GENERICAS:
        i += 1

    if i < len(tokens) and tokens[i].lower().strip(",.-'") in _PREPOSICIONES:
        candidato = " ".join(tokens[i + 1:])
    else:
        candidato = " ".join(tokens[i:])

    candidato = _RE_DE_CONTRACTA.sub("", candidato)
    candidato = candidato.strip(" ,.-")
    return candidato or texto


_candidatos = curses_limpio["nombre_carrera"].drop_duplicates().apply(_candidato_lugar)
print("Ejemplos nombre_carrera -> candidato_lugar:")
print(
    pd.DataFrame({
        "nombre_carrera": curses_limpio["nombre_carrera"].drop_duplicates(),
        "candidato_lugar": _candidatos,
    }).sample(15, random_state=0)
)

Ejemplos nombre_carrera -> candidato_lugar:
                                         nombre_carrera  \
1735              SWIMRUN EDREAMS CAP DE CREUS BY ZOGGS   
377                UNIRUN, LA CURSA DE LES UNIVERSITATS   
1634                       44A CURSA POPULAR VILADECANS   
1137                             24È CROS SANTA SUSANNA   
301                           SANT SILVESTRE DEL MASNOU   
65                42A MARXA POPULAR TRAIL ARENYS DE MAR   
428       1A CURSA SOLIDÀRIA MOSSOS D'ESQUADRA BADALONA   
1370  CURSA MORITZ SANT ANTONI - CAMPIONAT DE CATALUNYA   
1988                    10A CURSA DE LA DONA VILADECANS   
1869                              58 CROS DE GRANOLLERS   
1338                 12A CURSA DE CAP D’ANY DE SABADELL   
352                     1R CROS SANT CEBRIÀ DE VALLALTA   
1558                       27A MILLA URBANA DE RIPOLLET   
1402  CAMPIONAT DE CATALUNYA TRAIL PROMOCIÓ + OPEN O...   
625                           CROS CIUTAT DE CERDANYOLA   

           

In [10]:
# Geocodificación con Nominatim/OpenStreetMap (igual que en ccnorte), con
# checkpoint propio en championsxip_ubicaciones.csv. addressdetails=True nos da
# el municipio, la comarca ("county") y la provincia ya separados (más
# fiable que recortar el texto de loc.address a mano). "county" solo está
# bien cubierto en Catalunya/Galicia en OSM — para otras zonas suele
# salir vacío, no es un fallo del código.
#
# Si el candidato completo no se encuentra, reintentamos quitando palabras
# por delante (p.ej. "CREMATORRONS LLIÇÀ D'AMUNT" falla entero, pero
# "LLIÇÀ D'AMUNT" sí se geocodifica — el nombre de la carrera llevaba un
# apodo/marca delante del lugar real). No encogemos por debajo de 2
# palabras: una sola palabra suelta ("RACE", "CARLEMANY") puede coincidir
# por casualidad con un topónimo real sin relación con la carrera, y eso
# sería peor que no encontrar nada.
import csv
import time


def _geocode_con_reintentos(geolocator, candidato, pausa_segundos):
    from geopy.exc import GeopyError

    tokens = candidato.split()
    max_start = max(0, len(tokens) - 2) if len(tokens) > 1 else 0

    for start in range(max_start + 1):
        query = " ".join(tokens[start:])
        if not query:
            break
        try:
            loc = geolocator.geocode(
                f"{query}, España", exactly_one=True, country_codes="es",
                addressdetails=True, timeout=10,
            )
        except GeopyError:
            loc = None
        if loc:
            return loc
        if start < max_start:
            time.sleep(pausa_segundos)
    return None


def geocodificar_ubicaciones(nombres_carrera, out_dir, pausa_segundos: float = 1.1):
    from geopy.geocoders import Nominatim

    out_path = Path(out_dir)
    csv_ubic = out_path / "championsxip_ubicaciones.csv"

    cache = {}
    if csv_ubic.exists():
        prev = pd.read_csv(csv_ubic, dtype=str)
        cache = {row["nombre_carrera"]: row.to_dict() for _, row in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} carreras ya geocodificadas")

    geolocator = Nominatim(user_agent="championsxip_scraper_claudia")

    nombres_unicos = list(dict.fromkeys(n for n in nombres_carrera if isinstance(n, str)))
    pendientes = [n for n in nombres_unicos if n not in cache]
    print(f"Carreras a geocodificar: {len(pendientes)} (de {len(nombres_unicos)} únicas)")

    campos = ["nombre_carrera", "candidato_lugar", "municipio", "comarca", "provincia", "lat", "lon"]
    write_header = not csv_ubic.exists()
    with open(csv_ubic, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        if write_header:
            writer.writeheader()

        for i, nombre in enumerate(pendientes, 1):
            candidato = _candidato_lugar(nombre)
            fila = {c: None for c in campos}
            fila["nombre_carrera"] = nombre
            fila["candidato_lugar"] = candidato
            try:
                loc = _geocode_con_reintentos(geolocator, candidato, pausa_segundos)
                if loc:
                    addr = loc.raw.get("address", {})
                    fila["municipio"] = (
                        addr.get("city") or addr.get("town") or addr.get("village")
                        or addr.get("municipality")
                    )
                    fila["comarca"] = addr.get("county")
                    fila["provincia"] = addr.get("province") or addr.get("state")
                    fila["lat"] = loc.latitude
                    fila["lon"] = loc.longitude
            except Exception as e:
                print(f"  [{nombre}] ERROR inesperado: {e}")

            writer.writerow(fila)
            f.flush()
            cache[nombre] = fila

            if i % 25 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")
            time.sleep(pausa_segundos)  # respeta el límite de Nominatim (1 req/s)

    print(f"CSV de ubicaciones: {csv_ubic.resolve()}")
    df_ubic = pd.DataFrame(cache.values())
    print(f"Geocodificadas con éxito: {df_ubic['municipio'].notna().sum()} de {len(df_ubic)}")
    return df_ubic

### Esquema común entre las 10 fuentes

Para poder comparar o concatenar directamente las tablas de buscametas, championsxip, carreirasgalegas, ccnorte, cronofinisher, mychip, sportmaniacs, cursescat, iter5 y cruzandolameta, las 12 columnas que comparten todas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia` (antes `finished_d`/`finished_h` aquí, y sin ubicación). `fuente` es una constante ("championsxip") para identificar el origen al concatenar las 10 tablas. `municipio`/`comarca`/`provincia` se geocodifican más arriba.

In [11]:
OUT_DIR = Path("../../data/raw/championsxip")

df_ubicaciones = geocodificar_ubicaciones(curses_limpio["nombre_carrera"], out_dir=OUT_DIR)

curses_limpio = curses_limpio.merge(
    df_ubicaciones[["nombre_carrera", "municipio", "comarca", "provincia"]],
    on="nombre_carrera", how="left",
)

# Añadimos "fuente" (constante, para identificar el origen al concatenar
# con las otras 5 tablas) y "dia_semana" (derivado de "fecha"), y
# reordenamos las columnas para que el esquema común (fuente,
# nombre_carrera, fecha, dia_semana, distancia, tipo_modalidad, publico,
# finisher_d, finisher_h, municipio, comarca, provincia) quede igual en
# las 6 fuentes.
curses_limpio["fuente"] = "championsxip"

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

curses_limpio = curses_limpio[
    ["fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
     "finisher_d", "finisher_h", "municipio", "comarca", "provincia"]
]

print("Filas con municipio geocodificado:", curses_limpio["municipio"].notna().sum(),
      "de", len(curses_limpio))
curses_limpio.sample(15)

Checkpoint: 494 carreras ya geocodificadas
Carreras a geocodificar: 0 (de 494 únicas)
CSV de ubicaciones: /Users/claudiarm2002/Desktop/TFM/data/raw/championsxip/championsxip_ubicaciones.csv
Geocodificadas con éxito: 372 de 494
Filas con municipio geocodificado: 2184 de 2632


,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia
298,championsxip,13È CROS D’ALELLA,2021-12-19,Domingo,0.000,trail running,Infantil,17,27,Alella,NaN,Barcelona
1671,championsxip,CURSA FANNY SALLÉS – 5K TERRASSA,2024-10-06,Domingo,0.000,road running,Absoluta/General,0,247,NaN,NaN,NaN
1934,championsxip,SENDERS DE CAN COSTA,2025-02-09,Domingo,21.000,trail running,Absoluta/General,17,72,NaN,NaN,NaN
1024,championsxip,CROS ESCOLAR DE SANT ANDREU,2023-05-14,Domingo,0.000,trail running,Infantil,10,8,Barcelona,NaN,Barcelona
62,championsxip,XIV CROS ESCOLAR SANTS MONTJUÏC 2021,2021-05-09,Domingo,0.000,trail running,Infantil,9,10,Barcelona,NaN,Barcelona
748,championsxip,24 HORES ULTRAFONS EN PISTA,2022-12-17,Sábado,0.000,road running,Absoluta/General,0,0,Mota del Cuervo,NaN,Cuenca
1872,championsxip,58 CROS DE GRANOLLERS,2024-12-15,Domingo,0.000,trail running,Infantil,70,0,Granollers,NaN,Barcelona
1437,championsxip,XXÈ CROS DE L'EIXAMPLE 2024,2024-02-18,Domingo,0.000,trail running,Infantil,0,39,Barcelona,NaN,Barcelona
1207,championsxip,CROS POPULAR DE SANTS,2023-11-05,Domingo,1.609,road running,Absoluta/General,38,61,Barcelona,NaN,Barcelona
2111,championsxip,CURSA DE NOU BARRIS,2025-05-18,Domingo,5.000,road running,Absoluta/General,181,216,Barcelona,NaN,Barcelona


In [12]:
# Vista final de la tabla ya limpia y clasificada
print("Columnas:", list(curses_limpio.columns))
print("Filas x columnas:", curses_limpio.shape)
print()
print(curses_limpio.dtypes)
print()
print("Cruce tipo_modalidad x publico:")
print(pd.crosstab(curses_limpio["tipo_modalidad"], curses_limpio["publico"]))
print()
curses_limpio.sample(15)

Columnas: ['fuente', 'nombre_carrera', 'fecha', 'dia_semana', 'distancia', 'tipo_modalidad', 'publico', 'finisher_d', 'finisher_h', 'municipio', 'comarca', 'provincia']
Filas x columnas: (2632, 12)

fuente                    object
nombre_carrera            object
fecha             datetime64[ns]
dia_semana                object
distancia                float64
tipo_modalidad            object
publico                   object
finisher_d                 int64
finisher_h                 int64
municipio                 object
comarca                   object
provincia                 object
dtype: object

Cruce tipo_modalidad x publico:
publico         Absoluta/General  Elite  Equipos  Infantil  Mayores/Veteranos  \
tipo_modalidad                                                                  
road running                 828      4       23       439                 35   
trail running                395      0        0       869                 36   

publico         Otros  
tipo_moda

,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia
1873,championsxip,58 CROS DE GRANOLLERS,2024-12-15,Domingo,0.0,trail running,Infantil,0,82,Granollers,NaN,Barcelona
2416,championsxip,33È CROS ESCOLAR I 30È CROS POPULAR DE LES FRA...,2026-01-17,Sábado,0.0,road running,Infantil,15,22,NaN,NaN,NaN
590,championsxip,"MARATÓ TRAIL TOSSA DE MAR ""LA MEGALÍTICA""",2022-10-16,Domingo,5.0,trail running,Absoluta/General,53,50,Salorino,NaN,Cáceres
619,championsxip,CURSA EL PONT DE VILOMARA I ROCAFORT,2022-11-06,Domingo,5.0,trail running,Absoluta/General,24,30,el Pont de Vilomara i Rocafort,NaN,Barcelona
991,championsxip,26A MILLA URBANA DE RIPOLLET,2023-05-06,Sábado,0.0,road running,Infantil,18,0,Ripollet,NaN,Barcelona
1635,championsxip,44A CURSA POPULAR VILADECANS,2024-09-11,Miércoles,0.0,road running,Infantil,3,10,Viladecans,NaN,Barcelona
239,championsxip,55 CROS DE GRANOLLERS,2021-12-05,Domingo,0.0,trail running,Infantil,0,46,Granollers,NaN,Barcelona
1413,championsxip,27È CROS DE CANET,2024-02-04,Domingo,0.0,trail running,Infantil,34,0,Lleida,NaN,Lleida
391,championsxip,CROS ESCOLAR CIUTAT VELLA,2022-03-13,Domingo,0.0,trail running,Infantil,19,26,Sollana,NaN,València / Valencia
2160,championsxip,12A MILLA URBANA DE SANT PERE,2025-06-29,Domingo,0.0,road running,Infantil,8,6,Sant Pere de Ribes,NaN,Barcelona


In [13]:
# Guardamos la tabla ya limpia y clasificada para poder descargarla.
SALIDA = Path("../../data/processed/championsxip/DF_CHAMPIONSXIP_LIMPIO.csv")
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en ../../data/processed/championsxip/DF_CHAMPIONSXIP_LIMPIO.csv
